In [45]:
import pandas as pd
import numpy as np

sales = pd.read_csv("../data/processed/sales_clean.csv", parse_dates=["InvoiceDate"])
fc = pd.read_csv("../data/processed/final_forecast.csv", index_col=0, parse_dates=True).iloc[:, 0]
daily = pd.read_csv("../data/processed/daily_prepared.csv", index_col=0, parse_dates=True)["Revenue"]

train_end = pd.Timestamp("2011-11-08")
hist = sales[sales["InvoiceDate"] < train_end + pd.Timedelta(days=1)]

# Growth factor: forecast for the next 30 days vs the last 30 actual days
last30 = daily.loc[train_end - pd.Timedelta(days=29): train_end].sum()
growth = fc.sum() / last30
print("Forecast growth factor:", round(growth, 3))

Forecast growth factor: 1.176


In [46]:
start = train_end - pd.Timedelta(days=89)
recent = hist[hist["InvoiceDate"] >= start].copy()
recent["date"] = recent["InvoiceDate"].dt.normalize()
days = pd.date_range(start, train_end)

top = recent.groupby("StockCode")["Revenue"].sum().nlargest(30).index
units = (recent[recent["StockCode"].isin(top)]
         .groupby(["StockCode", "date"])["Quantity"].sum()
         .unstack(0).reindex(days).fillna(0))

desc = recent.groupby("StockCode")["Description"].agg(lambda s: s.mode().iloc[0])
price = recent.groupby("StockCode")["Price"].median()
print(units.shape)

(90, 30)


In [47]:
rng = np.random.default_rng(42)

inv = pd.DataFrame({
    "Description": desc.reindex(units.columns),
    "UnitPrice": price.reindex(units.columns),
    "AvgDailyDemand": units.mean() * growth,
    "DemandStd": units.std() * growth,
})

# SIMULATED inputs (not in the dataset)
inv["LeadTime"] = rng.integers(5, 15, len(inv))
inv["CurrentStock"] = (inv["AvgDailyDemand"] * inv["LeadTime"]
                       * rng.uniform(0.3, 3.0, len(inv))).round()

Z, ORDER_COST, HOLD_RATE = 1.65, 50, 0.20     # 95% service level, £50 per order, 20% holding cost

inv["SafetyStock"] = (Z * inv["DemandStd"] * np.sqrt(inv["LeadTime"])).round()
inv["ReorderPoint"] = (inv["AvgDailyDemand"] * inv["LeadTime"] + inv["SafetyStock"]).round()
annual = inv["AvgDailyDemand"] * 365
inv["EOQ"] = np.sqrt(2 * annual * ORDER_COST / (HOLD_RATE * inv["UnitPrice"])).round()

def status(r):
    if r["CurrentStock"] <= r["ReorderPoint"]:
        return "Reorder"
    if r["CurrentStock"] > r["ReorderPoint"] + r["EOQ"]:
        return "Overstock"
    return "Healthy"

inv["Status"] = inv.apply(status, axis=1)
inv["OrderQty"] = np.where(inv["Status"] == "Reorder",
                           (inv["ReorderPoint"] + inv["EOQ"] - inv["CurrentStock"]).clip(lower=0), 0)

print(inv["Status"].value_counts())
inv.round(1).head(10)

Status
Reorder    19
Healthy    11
Name: count, dtype: int64


,Description,UnitPrice,AvgDailyDemand,DemandStd,LeadTime,CurrentStock,SafetyStock,ReorderPoint,EOQ,Status,OrderQty
StockCode,,,,,,,,,,,
21915,RED HARMONICA IN BOX,1.2,136.6,356.0,5,624.0,1313.0,1996.0,4466.0,Reorder,5838.0
22086,PAPER CHAIN KIT 50'S CHRISTMAS,3.0,105.4,186.5,12,2274.0,1066.0,2331.0,2554.0,Reorder,2611.0
22111,SCOTTIE DOG HOT WATER BOTTLE,5.0,24.3,69.5,11,126.0,380.0,648.0,947.0,Reorder,1469.0
22112,CHOCOLATE HOT WATER BOTTLE,5.0,27.8,33.2,9,635.0,164.0,414.0,1013.0,Healthy,0.0
22114,HOT WATER BOTTLE TEA AND SYMPATHY,4.2,31.5,55.5,9,568.0,275.0,558.0,1163.0,Healthy,0.0
22138,BAKING SET 9 PIECE RETROSPOT,5.0,24.8,56.1,13,757.0,334.0,656.0,956.0,Healthy,0.0
22197,POPCORN HOLDER,0.8,204.0,390.5,5,1283.0,1441.0,2461.0,6619.0,Reorder,7797.0
22386,JUMBO BAG PINK POLKADOT,2.1,103.8,160.8,11,3334.0,880.0,2022.0,3018.0,Healthy,0.0
22423,REGENCY CAKESTAND 3 TIER,12.8,34.9,40.2,7,663.0,175.0,419.0,707.0,Healthy,0.0


In [48]:
inv.reset_index().to_csv("../data/processed/inventory_recommendations.csv", index=False)
print("Saved", inv.shape)

Saved (30, 11)


In [49]:
test_days = pd.date_range("2011-11-09", "2011-12-08")
t = sales[(sales["InvoiceDate"] >= test_days[0]) &
          (sales["InvoiceDate"] < test_days[-1] + pd.Timedelta(days=1))].copy()
t["date"] = t["InvoiceDate"].dt.normalize()

actual = (t[t["StockCode"].isin(inv.index)]
          .groupby(["StockCode", "date"])["Quantity"].sum()
          .unstack(0).reindex(test_days).fillna(0))
actual = actual.reindex(columns=inv.index).fillna(0)
print(actual.shape)

(30, 30)


In [50]:
def simulate(inv, actual, policy):
    out = []
    for sku, r in inv.iterrows():
        rop, qty = policy(r)
        stock, pipeline = r["CurrentStock"], []
        unmet, stock_sum = 0.0, 0.0
        for i, d in enumerate(actual.index):
            stock += sum(q for a, q in pipeline if a == i)
            pipeline = [(a, q) for a, q in pipeline if a != i]
            demand = actual.loc[d, sku]
            sold = min(stock, demand)
            unmet += demand - sold
            stock -= sold
            position = stock + sum(q for _, q in pipeline)
            if position <= rop:
                pipeline.append((i + int(r["LeadTime"]), qty))
            stock_sum += stock
        out.append({
            "StockCode": sku,
            "Demand": actual[sku].sum(),
            "Unmet": unmet,
            "AvgInvValue": stock_sum / len(actual) * r["UnitPrice"],
            "UnitPrice": r["UnitPrice"],
        })
    return pd.DataFrame(out).set_index("StockCode")

naive_policy = lambda r: (r["AvgDailyDemand"] * r["LeadTime"], 14 * r["AvgDailyDemand"])
opt_policy = lambda r: (r["ReorderPoint"], r["EOQ"])

res_naive = simulate(inv, actual, naive_policy)
res_opt = simulate(inv, actual, opt_policy)

HOLD = 0.20 * 30 / 365      # holding cost for 30 days
MARGIN = 0.40               # ASSUMED margin lost on an unmet sale

def summarize(res):
    unmet_pct = res["Unmet"].sum() / res["Demand"].sum() * 100
    inv_val = res["AvgInvValue"].sum()
    lost = (res["Unmet"] * res["UnitPrice"] * MARGIN).sum()
    hold = inv_val * HOLD
    return pd.Series({"Unmet demand %": unmet_pct, "Avg inventory £": inv_val,
                      "Lost margin £": lost, "Holding cost £": hold,
                      "Total cost £": lost + hold})

summary = pd.DataFrame({"Naive": summarize(res_naive), "Optimized": summarize(res_opt)}).round(1)
summary["Change %"] = ((summary["Optimized"] - summary["Naive"]) / summary["Naive"] * 100).round(1)
summary

,Naive,Optimized,Change %
Unmet demand %,28.1,12.9,-54.1
Avg inventory £,48003.4,105737.2,120.3
Lost margin £,27591.0,11744.6,-57.4
Holding cost £,789.1,1738.1,120.3
Total cost £,28380.1,13482.7,-52.5


In [51]:
summary.to_csv("../data/processed/inventory_policy_comparison.csv")